# YOLO 物件偵測訓練

In [1]:
PRETRAIN_MODEL = "yolo11n.pt"
LABEL_DIR = "/media/jianhua/HDD 1T/mot_annotations"
DATASET_DIR = "/home/jianhua/Desktop/dataset/SeaDronesSee_MOT"
YOLO_LABEL_DIR = "/media/jianhua/HDD 1T/yolo_obj_detect"

## 轉換格式 COCO -> YOLO

In [7]:
from ultralytics.data.converter import convert_coco

convert_coco(
    labels_dir=LABEL_DIR,
    save_dir=YOLO_LABEL_DIR
)

Annotations /media/jianhua/HDD 1T/mot_annotations/instances_train_objects_in_water.json: 100% ━━━━━━━━━━━━ 27226/27226 8.4Kit/s 3.2s0.0ss
Annotations /media/jianhua/HDD 1T/mot_annotations/instances_val_objects_in_water.json: 100% ━━━━━━━━━━━━ 8579/8579 8.2Kit/s 1.0s0.0s
COCO data converted successfully.
Results saved to /media/jianhua/HDD 1T/yolo_obj_detect


In [8]:
import os
import shutil

src_labels = os.path.join(YOLO_LABEL_DIR, "labels")
dst_labels = os.path.join(DATASET_DIR, "labels")

# 1️⃣ 如果 DATASET_DIR 已有 labels 就刪掉
if os.path.exists(dst_labels):
    shutil.rmtree(dst_labels)

# 2️⃣ 搬移 labels 資料夾
shutil.move(src_labels, dst_labels)

# 3️⃣ 刪掉整個 YOLO_LABEL_DIR
if os.path.exists(YOLO_LABEL_DIR):
    shutil.rmtree(YOLO_LABEL_DIR)

print("labels moved and YOLO_LABEL_DIR removed.")

# ================= 新增的部分 =================

# 定義 classes.txt 的來源與目標路徑
src_classes = "classes.txt" # 當下目錄下的 classes.txt
dst_train_dir = os.path.join(DATASET_DIR, "labels", "train")
dst_val_dir = os.path.join(DATASET_DIR, "labels", "val")

# 確保目標資料夾 train 和 val 存在 (避免 shutil.copy 找不到資料夾報錯)
os.makedirs(dst_train_dir, exist_ok=True)
os.makedirs(dst_val_dir, exist_ok=True)

# 4️⃣ 複製 classes.txt 到 train 與 val 資料夾
if os.path.exists(src_classes):
    shutil.copy(src_classes, dst_train_dir)
    shutil.copy(src_classes, dst_val_dir)
    print("classes.txt successfully copied to train and val directories.")
else:
    print(f"⚠️ 警告：在當下目錄找不到 {src_classes}，略過複製步驟。")

labels moved and YOLO_LABEL_DIR removed.
classes.txt successfully copied to train and val directories.


## 開始訓練

In [9]:
from ultralytics import YOLO
import os

model = YOLO(PRETRAIN_MODEL)  # load a pretrained model (recommended for training)

val_classes = os.path.join(DATASET_DIR, "labels/val/classes.txt")
train_classes = os.path.join(DATASET_DIR, "labels/train/classes.txt")

if os.path.exists(train_classes) and os.path.exists(val_classes):
    results = model.train(data="data.yaml", epochs=10, imgsz=640, batch=32)
else:
    raise FileNotFoundError(f"Missing required files or directories: {val_classes} {train_classes}")

New https://pypi.org/project/ultralytics/8.4.23 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.21 🚀 Python-3.9.25 torch-2.0.0 CUDA:0 (NVIDIA GeForce RTX 4070, 11866MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, 

/home/jianhua/miniconda3/envs/torch2/lib/python3.9/site-packages/ultralytics/utils/metrics.py:839: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/home/jianhua/miniconda3/envs/torch2/lib/python3.9/site-packages/numpy/core/_methods.py:182: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/10      4.36G          0      2.228          0          0        640: 7% ╸─────────── 56/852 1.8it/s 36.4s<7:33


KeyboardInterrupt: 

# 驗證 Track 效果 (HOTA, MOTA...)

In [7]:
MODEL = "runs/detect/train18/weights/best.pt"
GT_JSON = "/media/jianhua/HDD 1T/mot_annotations/instances_val_objects_in_water.json"
IMAGE_FOLDER = "/home/jianhua/Desktop/dataset/SeaDronesSee_MOT/images/val"

In [8]:
from ultralytics import YOLO

model = YOLO(MODEL) # Load a pretrained YOLO model

## 建立 ID 和 圖片路徑 對照表

In [9]:
import json
import os

def create_image_path_mapping(json_file_path, image_folder):
    """
    讀取 COCO JSON 檔案，將 image_id 與完整的圖片路徑對應起來。
    """
    with open(json_file_path, 'r', encoding='utf-8') as f:
        coco_data = json.load(f)

    # 用來存放您要求的字典格式 (List of Dicts)
    # 格式: [{"id": 1, "path": "/Volumes/.../001.jpg"}, ...]
    formatted_list = []
    
    # 💡 額外加碼：用來快速查詢的 Mapping Dictionary (實務上更常用)
    # 格式: {1: "/Volumes/.../001.jpg", 2: ...}
    id_to_path_dict = {}

    for img_info in coco_data.get('images', []):
        # 兼容標準 COCO ('id', 'file_name') 與您自訂的 key ('image_id', 'image_name')
        img_id = img_info.get('image_id', img_info.get('id'))
        img_name = img_info.get('image_name', img_info.get('file_name').replace(".png",".jpg"))
        
        if img_id is not None and img_name is not None:
            # 安全地組合資料夾與檔名
            full_path = os.path.join(image_folder, img_name)
            
            # 1. 建立您要求的獨立 dict 結構
            formatted_list.append({"id": img_id, "path": full_path})
            
            # 2. 建立 O(1) 快速查詢的字典
            id_to_path_dict[img_id] = full_path

    return formatted_list, id_to_path_dict

In [10]:
# 執行函數
dict_list, mapping_dict = create_image_path_mapping(GT_JSON, IMAGE_FOLDER)

# 檢視結果 (印出前 3 筆作為範例)
print("--- 1. 您要求的字典列表格式 ---")
for item in dict_list[:3]:
    print(item)
    
print("\n--- 2. 實務上便於查詢的 Mapping 字典 ---")
# 假設我們要查 ID 為 1 的圖片路徑 (如果有的話)
sample_id = dict_list[0]['id'] if dict_list else None
if sample_id is not None:
    print(f"ID {sample_id} 的路徑是: {mapping_dict[sample_id]}")

--- 1. 您要求的字典列表格式 ---
{'id': 267, 'path': '/home/jianhua/Desktop/dataset/SeaDronesSee_MOT/images/val/267.jpg'}
{'id': 268, 'path': '/home/jianhua/Desktop/dataset/SeaDronesSee_MOT/images/val/268.jpg'}
{'id': 269, 'path': '/home/jianhua/Desktop/dataset/SeaDronesSee_MOT/images/val/269.jpg'}

--- 2. 實務上便於查詢的 Mapping 字典 ---
ID 267 的路徑是: /home/jianhua/Desktop/dataset/SeaDronesSee_MOT/images/val/267.jpg


## COCO GT File -> Track Eval File

In [11]:
OUT_GT_TXT = "trackeval/gt.txt"

In [12]:
import json

with open(GT_JSON) as f:
    data = json.load(f)

# 1. 取得所有 category 的 id 並進行排序
# 以你的資料為例，取出來排序後會是 [1, 2, 3, 6]
original_cat_ids = sorted([cat["id"] for cat in data["categories"]])

# 2. 建立新舊 ID 對應表 (將其對應到 0 ~ num_classes-1)
# 結果會是類似這樣：{1: 0, 2: 1, 3: 2, 6: 3}
cat_id_map = {old_id: new_id for new_id, old_id in enumerate(original_cat_ids)}

# (選擇性) 如果你想確認對應關係，可以印出來看看
print("Category ID 轉換對應表:", cat_id_map)

rows = []

for ann in data["annotations"]:
    frame = ann["image_id"]
    tid = ann["track_id"]
    
    # 3. 透過對應表取得新的 class id
    original_cat_id = ann["category_id"]
    new_cat_id = cat_id_map[original_cat_id]
    
    x, y, w, h = ann["bbox"]

    # 注意這邊將原本的 cat_id 替換成 new_cat_id
    rows.append(f"{frame},{tid},{x},{y},{w},{h},1,{new_cat_id},1")

with open(OUT_GT_TXT, "w") as f:
    f.write("\n".join(rows))
    
print("轉換完成！")

Category ID 轉換對應表: {1: 0, 2: 1, 3: 2, 6: 3}
轉換完成！


## 測試追蹤

In [20]:
OUT_PREDICT_TXT = "trackeval/predict.txt"
TRACKER_TYPE =  "botsort" # "bytetrack"

In [21]:
import time
from collections import defaultdict
import cv2
import numpy as np
from ultralytics import YOLO

def obj_track_pipeline(model: YOLO, 
                       video_path: str = None, 
                       image_list: list = None, 
                       output_video_path: str = None, 
                       output_txt_path: str = None,
                       tracker_type: str = "bytetrack",
                       video_fps: int = 30):
    """
    執行 YOLO 物件追蹤，支援影片或圖片序列輸入，並可選擇輸出影片、MOT 格式的 txt 檔，或兩者皆輸出。
    
    參數:
        model: YOLO 模型實例
        video_path: 輸入影片路徑 (若使用圖片序列請設為 None)
        image_list: 輸入圖片序列 dict (格式: [{"path": "...", "id": ...}])
        output_video_path: 輸出追蹤影片的儲存路徑 (若不輸出請設為 None)
        output_txt_path: 輸出 txt 檔的儲存路徑 (若不輸出請設為 None)
        tracker_type: 追蹤器種類，"bytetrack" 或 "botsort"
        video_fps: 當輸入為圖片序列且需輸出影片時，指定的影片 FPS 幀率
    """
    
    # --- 1. 輸入與輸出檢查 ---
    if not video_path and not image_list:
        raise ValueError("❌ 錯誤：必須提供 video_path 或 image_list 其中之一！")
    if not output_video_path and not output_txt_path:
        raise ValueError("❌ 錯誤：至少需要指定 output_video_path 或 output_txt_path 進行輸出！")

    global_start_time = time.time()
    yolo_total_time = 0
    current_frame = 0
    output_lines = []

    # --- 2. 初始化輸入模式 ---
    if video_path:
        mode = "video"
        cap = cv2.VideoCapture(video_path)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        video_fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f"🎥 開始處理影片: {video_path}")
    else:
        mode = "image"
        total_frames = len(image_list)
        print(f"🖼️ 開始處理圖片序列 (共 {total_frames} 張)")
        
        # 若輸入是圖片且要輸出影片，需要先讀取第一張圖來取得長寬
        if output_video_path and total_frames > 0:
            first_img = cv2.imread(image_list[0]["path"])
            if first_img is not None:
                height, width = first_img.shape[:2]
            else:
                height, width = 640, 480 # 預設值防錯

    # --- 3. 初始化輸出模式 ---
    if output_video_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video_writer = cv2.VideoWriter(output_video_path, fourcc, video_fps, (width, height))
        track_history = defaultdict(lambda: [])
        print(f"🎬 準備輸出影片至: {output_video_path}")
        
    if output_txt_path:
        print(f"📝 準備輸出 Tracking TXT 至: {output_txt_path}")

    tracker_config = "bytetrack.yaml" if tracker_type == "bytetrack" else "botsort.yaml"

    # --- 4. 主迴圈開始 ---
    while True:
        # 取得下一幀與 ID
        if mode == "video":
            success, frame = cap.read()
            if not success:
                break
            current_frame += 1
            output_id = current_frame
        else:
            if current_frame >= total_frames:
                break
            item = image_list[current_frame]
            img_path = item["path"]
            output_id = item.get("id", current_frame + 1) # 防錯: 如果沒給id預設用幀數
            
            frame = cv2.imread(img_path)
            current_frame += 1
            
            if frame is None:
                print(f"\n⚠️ 警告：無法讀取圖片 {img_path}，略過此幀。")
                continue

        # YOLO 追蹤推論
        yolo_start = time.time()
        results = model.track(frame, persist=True, verbose=False, device="cuda", tracker=tracker_config)
        yolo_end = time.time()
        yolo_total_time += (yolo_end - yolo_start)

        # 解析結果
        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xywh.cpu().numpy()
            track_ids = results[0].boxes.id.int().cpu().tolist()
            classes = results[0].boxes.cls.int().cpu().tolist()

            # 處理影片輸出 (畫框與軌跡)
            if output_video_path:
                annotated_frame = results[0].plot()
                for box, track_id in zip(boxes, track_ids):
                    x_center, y_center, w, h = box
                    track = track_history[track_id]
                    track.append((float(x_center), float(y_center)))
                    
                    if len(track) > 30:
                        track.pop(0)
                        
                    points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                    cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 230, 230), thickness=5)
                
                video_writer.write(annotated_frame)

            # 處理 TXT 輸出 (轉換為 MOT 格式)
            if output_txt_path:
                for box, track_id, cls in zip(boxes, track_ids, classes):
                    x_center, y_center, w, h = box
                    # YOLO center bbox → MOT top-left bbox
                    x = int(x_center - w / 2)
                    y = int(y_center - h / 2)
                    line = f"{output_id},{track_id},{x},{y},{int(w)},{int(h)},1,{cls},1"
                    output_lines.append(line)
        else:
            # 如果這幀沒抓到物件，且需要寫入影片，就直接寫入原圖確保影片連續性
            if output_video_path:
                video_writer.write(frame)

        # 計算與顯示 FPS
        elapsed_time = time.time() - global_start_time
        avg_fps = current_frame / elapsed_time if elapsed_time > 0 else 0.0
        yolo_avg_fps = current_frame / yolo_total_time if yolo_total_time > 0 else 0.0
        
        print(f"\r進度: {current_frame}/{total_frames} | YOLO FPS: {yolo_avg_fps:.1f} | Total FPS: {avg_fps:.1f}", end="", flush=True)

    # --- 5. 釋放資源與存檔 ---
    print("\n")
    if mode == "video":
        cap.release()
        
    if output_video_path:
        video_writer.release()
        print(f"✅ 追蹤影片已成功匯出至: {output_video_path}")
        
    if output_txt_path:
        with open(output_txt_path, "w") as f:
            f.write("\n".join(output_lines))
        print(f"✅ Tracking結果 txt 已輸出至: {output_txt_path}")

    total_time = time.time() - global_start_time
    print(f"⏱️ 總耗時: {total_time:.1f} 秒\n")

In [ ]:
# 執行追蹤並匯出影片
# frames_obj_track(
#     model=model, 
#     video_path="/Volumes/Transcend/Compressed/video/DJI_0001.mov", 
#     output_path="tracked_output.mp4"
# ) 

obj_track_pipeline(model, image_list=dict_list, output_txt_path=OUT_PREDICT_TXT, output_video_path="/Volumes/Transcend/Compressed/track_out.mp4", tracker_type=TRACKER_TYPE)

🖼️ 開始處理圖片序列 (共 8584 張)
進度: 8584/8584 | YOLO FPS: 13.0 | Total FPS: 10.4
✅ Tracking結果已輸出至: trackeval/predict.txt
⏱️ 總耗時: 821.9 秒



## 計算追蹤指標

In [23]:
YAML_PATH = "data.yaml"  # 替換成你的 data.yaml 實際路徑
GT_TXT = "trackeval/gt.txt"
PRED_TXT = "trackeval/predict.txt"

In [24]:
import yaml

# 讀取 yaml 檔案
with open(YAML_PATH, 'r', encoding='utf-8') as f:
    yaml_data = yaml.safe_load(f)

# 直接將 names 寫入 GT_CLASSES 變數
GT_CLASSES = yaml_data.get('names', {})

# 印出來確認結果
print(GT_CLASSES)

{0: 'swimmer', 1: 'swimmer_with_life_jacket', 2: 'boat', 3: 'life_jacket'}


In [25]:
def write_seqinfo(seq_dir, seq_name, max_frame):
    seqinfo = (
        "[Sequence]\n"
        f"name={seq_name}\n"
        "imDir=img1\n"
        "frameRate=30\n"
        f"seqLength={max_frame}\n"
        "imWidth=3840\n"
        "imHeight=2160\n"
        "imExt=.jpg\n"
    )
    with open(f"{seq_dir}/seqinfo.ini", "w") as f:
        f.write(seqinfo)

In [26]:
import os
import shutil
import pandas as pd
import trackeval
from ultralytics import YOLO

WORK_DIR = "trackeval_tmp"
SEQ = "seq1"
TRACKER = "tracker1"

# ===== 1. 建立類別名稱到 Predict ID 的對應表 =====
print("🔍 正在載入模型以獲取預測類別資訊...")
model = YOLO(MODEL)
# model.names 會回傳類似 {0: 'swimmer', 1: 'boat', 2: 'jetski'}
pred_names_dict = model.names 

# 反轉字典，變成以名稱查 ID：{'swimmer': 0, 'boat': 1, ...}
name_to_pred_id = {v: k for k, v in pred_names_dict.items()}

print("\n📊 類別匹配狀況:")
for gt_id, name in GT_CLASSES.items():
    pred_id = name_to_pred_id.get(name, "無對應")
    print(f"  - [{name}] GT ID: {gt_id} -> Predict ID: {pred_id}")

# ===== 2. 讀取資料 =====
try:
    gt = pd.read_csv(GT_TXT, header=None)
    pred = pd.read_csv(PRED_TXT, header=None)
except FileNotFoundError as e:
    print(f"\n❌ 找不到檔案: {e.filename}。請確認檔案是否存在。")
    exit()

gt.columns = ["frame", "id", "x", "y", "w", "h", "conf", "class", "vis"]
pred.columns = ["frame", "id", "x", "y", "w", "h", "score", "class", "misc"]

results_summary = {}

# ===== 3. 每個 class evaluation =====
for gt_class_id, class_name in GT_CLASSES.items():

    print(f"\n{'='*40}")
    print(f"🚀 開始評估類別: {class_name}")
    print(f"{'='*40}")

    # 檢查該類別是否在模型的預測清單中
    if class_name not in name_to_pred_id:
        print(f"⚠️ 類別 '{class_name}' 不在模型的預測能力內 (Model 中無此類別)，略過評估。")
        continue

    # 取得對應的預測 ID
    pred_class_id = name_to_pred_id[class_name]

    # 💡 核心邏輯：用 GT ID 篩選 GT 資料，用 Predict ID 篩選預測資料
    gt_cls = gt[gt["class"] == gt_class_id].copy()
    pred_cls = pred[pred["class"] == pred_class_id].copy()

    if len(gt_cls) == 0:
        print(f"⚠️ 類別 {class_name} 在 Ground Truth 中沒有資料，略過評估。")
        continue

    # 🧹 資料清洗 (剔除同幀重複的 ID 避免 TrackEval 報錯)
    gt_dups = gt_cls.duplicated(subset=["frame", "id"])
    if gt_dups.any():
        print(f"🧹 清理: GT 標註中發現 {gt_dups.sum()} 筆重複 (frame, id)，已剔除。")
        gt_cls = gt_cls.drop_duplicates(subset=["frame", "id"], keep="first")

    pred_dups = pred_cls.duplicated(subset=["frame", "id"])
    if pred_dups.any():
        print(f"🧹 清理: 預測結果中發現 {pred_dups.sum()} 筆重複 (frame, id)，已剔除。")
        pred_cls = pred_cls.drop_duplicates(subset=["frame", "id"], keep="first")

    # TrackEval 預設只評估 'pedestrian' (class=1)，所以把篩選出來的資料都改成 1
    gt_cls["class"] = 1
    pred_cls["class"] = 1

    # ===== 建立資料夾結構 =====
    gt_seq_dir = f"{WORK_DIR}/gt/{SEQ}"
    gt_txt_dir = f"{gt_seq_dir}/gt"
    tr_dir = f"{WORK_DIR}/trackers/{TRACKER}/data"

    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
        
    os.makedirs(gt_txt_dir, exist_ok=True)
    os.makedirs(tr_dir, exist_ok=True)

    gt_cls.to_csv(f"{gt_txt_dir}/gt.txt", header=False, index=False)
    pred_cls.to_csv(f"{tr_dir}/{SEQ}.txt", header=False, index=False)

    max_frame = int(gt_cls["frame"].max())
    write_seqinfo(gt_seq_dir, SEQ, max_frame)

    # ===== TrackEval Config 設定 =====
    eval_config = trackeval.Evaluator.get_default_eval_config()
    dataset_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()

    eval_config["PRINT_CONFIG"] = False
    eval_config["PRINT_RESULTS"] = False
    eval_config["DISPLAY_LESS_PROGRESS"] = True

    dataset_config["GT_FOLDER"] = f"{WORK_DIR}/gt"
    dataset_config["TRACKERS_FOLDER"] = f"{WORK_DIR}/trackers"
    dataset_config["TRACKERS_TO_EVAL"] = [TRACKER]
    dataset_config["SKIP_SPLIT_FOL"] = True 
    dataset_config["SEQ_INFO"] = {SEQ: None}
    dataset_config["CLASSES_TO_EVAL"] = ["pedestrian"]

    evaluator = trackeval.Evaluator(eval_config)
    dataset = trackeval.datasets.MotChallenge2DBox(dataset_config)

    metrics = [
        trackeval.metrics.HOTA(),
        trackeval.metrics.CLEAR(),
        trackeval.metrics.Identity()
    ]

    results, _ = evaluator.evaluate([dataset], metrics)
    res = results["MotChallenge2DBox"][TRACKER][SEQ]["pedestrian"]

    results_summary[class_name] = {
        "HOTA": res["HOTA"]["HOTA"],
        "MOTA": res["CLEAR"]["MOTA"],
        "IDF1": res["Identity"]["IDF1"]
    }

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

# ===== 輸出結果 =====
print("\n" + "*"*40)
print("* FINAL METRICS SUMMARY       *")
print("*"*40)
 

print("\n========== FINAL METRICS ==========")
for cls, r in results_summary.items():
    print(f"\nClass: {cls}")

    hota = r['HOTA']
    mota = r['MOTA']
    idf1 = r['IDF1']

    # 如果是 numpy array，就取平均
    if hasattr(hota, "mean"):
        hota = hota.mean()
    if hasattr(mota, "mean"):
        mota = mota.mean()
    if hasattr(idf1, "mean"):
        idf1 = idf1.mean()

    print(f"HOTA : {hota:.2f}")
    print(f"MOTA : {mota:.2f}")
    print(f"IDF1 : {idf1:.2f}")
    print("-" * 20)

🔍 正在載入模型以獲取預測類別資訊...

📊 類別匹配狀況:
  - [swimmer] GT ID: 0 -> Predict ID: 0
  - [swimmer_with_life_jacket] GT ID: 1 -> Predict ID: 1
  - [boat] GT ID: 2 -> Predict ID: 2
  - [life_jacket] GT ID: 3 -> Predict ID: 3

🚀 開始評估類別: swimmer
🧹 清理: GT 標註中發現 6 筆重複 (frame, id)，已剔除。

MotChallenge2DBox Config:
GT_FOLDER            : trackeval_tmp/gt              
TRACKERS_FOLDER      : trackeval_tmp/trackers        
OUTPUT_FOLDER        : None                          
TRACKERS_TO_EVAL     : ['tracker1']                  
CLASSES_TO_EVAL      : ['pedestrian']                
BENCHMARK            : MOT17                         
SPLIT_TO_EVAL        : train                         
INPUT_AS_ZIP         : False                         
PRINT_CONFIG         : True                          
DO_PREPROC           : True                          
TRACKER_SUB_FOLDER   : data                          
OUTPUT_SUB_FOLDER    :                               
TRACKER_DISPLAY_NAMES : None                          
SEQ